# RAG Knowledge Base Builder (OpenAI/Euron Version)

This notebook is responsible for compiling and updating the RAG knowledge base for your portfolio's chatbot using OpenAI/Euron API. It processes project details, experiences, and blog data from YAML configuration files, parses local resume PDFs, generates dense vector embeddings using OpenAI's `text-embedding-3-small` model, and saves the final vector database to disk.

In [ ]:
# 1. Install Required Python Packages
!pip install -q pypdf langchain langchain-community python-dotenv pyyaml openai

In [ ]:
# 2. Initialize Directories and Environment Configurations
import os
import yaml
import time
import json
import glob
from dotenv import load_dotenv

# Load environment variables from the root .env file
load_dotenv()

# Target Directory Paths
resumes_in_dir = "data/resumes"
knowledge_dir = "data/knowledge"
vector_out_dir = "public/data/knowledge"

# Create directories automatically if they do not exist
os.makedirs(resumes_in_dir, exist_ok=True)
os.makedirs(knowledge_dir, exist_ok=True)
os.makedirs(vector_out_dir, exist_ok=True)


In [ ]:
from dotenv import load_dotenv
import os

# Load .env
load_dotenv(override=True)

# Read environment variables
api_key = os.getenv("VITE_OPENAI_API_KEY", "").strip()
base_url = os.getenv(
    "VITE_OPENAI_BASE_URL",
    "https://api.euron.one/api/v1/euri"
).strip()

if not api_key:
    raise ValueError(
        "❌ VITE_OPENAI_API_KEY not found in .env"
    )

print(f"✅ API Key Loaded : {api_key[:10]}...{api_key[-6:]}")
print(f"✅ Base URL       : {base_url}")

✅ API Key Loaded : euri-e7401...928602
✅ Base URL       : https://api.euron.one/api/v1/euri


In [2]:
# 3. Extract and Consolidate Text from YAML Files
yaml_sources = [
    ("src/data/portfolio.yaml", "Portfolio Profile"),
    ("src/data/projects.yaml", "Projects and Tech Stack"),
    ("src/data/blog.yaml", "Blogs and Articles")
]

consolidated_blocks = []

for filepath, name in yaml_sources:
    if os.path.exists(filepath):
        print(f"Parsing {name} from {filepath}...")
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                content = yaml.safe_load(f)
                # Convert YAML content to structured readable text
                text_block = f"=== CONTENT SOURCE: {name.upper()} ===\n"
                text_block += yaml.dump(content, default_flow_style=False, allow_unicode=True)
                consolidated_blocks.append(text_block)
        except Exception as e:
            print(f"❌ Error parsing {filepath}: {e}")
    else:
        print(f"⚠️ Warning: {filepath} not found. Skipping.")

knowledge_base_path = os.path.join(knowledge_dir, "knowledge_base.txt")
with open(knowledge_base_path, "w", encoding="utf-8") as f:
    f.write("\n\n".join(consolidated_blocks))

print(f"\n✅ Created consolidated knowledge text file at: {knowledge_base_path} ({len(consolidated_blocks)} sections parsed)")

Parsing Portfolio Profile from src/data/portfolio.yaml...
Parsing Projects and Tech Stack from src/data/projects.yaml...
Parsing Blogs and Articles from src/data/blog.yaml...

✅ Created consolidated knowledge text file at: data/knowledge\knowledge_base.txt (3 sections parsed)


In [ ]:
print(repr(api_key))
print(repr(base_url))

In [3]:
# 4. Load PDFs from data/resumes/
from langchain_community.document_loaders import PyPDFLoader

pdf_paths = glob.glob(os.path.join(resumes_in_dir, "*.pdf"))
resume_documents = []

if not pdf_paths:
    print("\n" + "!"*90)
    print("⚠️ No resume PDFs were found in the data/resumes folder.")
    print("Please add one or more resume PDF files before building the RAG knowledge base.")
    print("!"*90 + "\n")
else:
    print(f"Found {len(pdf_paths)} PDF(s) in {resumes_in_dir}. Parsing...")
    for path in pdf_paths:
        filename = os.path.basename(path)
        try:
            print(f"Loading {filename} via PyPDFLoader...")
            loader = PyPDFLoader(path)
            pages = loader.load()
            print(f"└─ Loaded {len(pages)} pages")
            resume_documents.extend(pages)
        except Exception as e:
            print(f"❌ Error loading {filename}: {e}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_22444\4234336313.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
⚠️ No resume PDFs were found in the data/resumes folder.
Please add one or more resume PDF files before building the RAG knowledge base.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!



In [4]:
# 5. Create Text Chunks from Sources
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunks_list = []

# Split YAML knowledge base
if os.path.exists(knowledge_base_path):
    with open(knowledge_base_path, "r", encoding="utf-8") as f:
        kb_content = f.read()
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=100
    )
    yaml_chunks = text_splitter.create_documents(
        [kb_content],
        metadatas=[{"source": "yaml_knowledge_base"}]
    )
    chunks_list.extend(yaml_chunks)
    print(f"Split knowledge_base.txt into {len(yaml_chunks)} chunks.")

# Split PDF resumes
if resume_documents:
    pdf_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100
    )
    pdf_chunks = pdf_splitter.split_documents(resume_documents)
    chunks_list.extend(pdf_chunks)
    print(f"Split resumes PDFs into {len(pdf_chunks)} chunks.")

print(f"\n✅ Total chunks prepared for embedding: {len(chunks_list)}")

Split knowledge_base.txt into 94 chunks.

✅ Total chunks prepared for embedding: 94


In [8]:
# ============================================================
# Dataset Statistics
# ============================================================

total_chunks = len(valid_chunks)
total_characters = 0
total_words = 0
unique_pages = set()
unique_sources = set()

for chunk in valid_chunks:
    text = chunk.page_content.strip()

    total_characters += len(text)
    total_words += len(text.split())

    metadata = chunk.metadata or {}

    if "page" in metadata:
        unique_pages.add(metadata["page"])

    if "source" in metadata:
        unique_sources.add(metadata["source"])

avg_chars = total_characters / total_chunks if total_chunks else 0
avg_words = total_words / total_chunks if total_chunks else 0

print("\n" + "=" * 70)
print("📊 DATASET SUMMARY")
print("=" * 70)
print(f"📄 Total Chunks      : {total_chunks:,}")
print(f"📝 Total Characters : {total_characters:,}")
print(f"🔤 Total Words      : {total_words:,}")

if unique_pages:
    print(f"📚 Total PDF Pages  : {max(unique_pages) + 1}")

if unique_sources:
    print(f"📂 Source Files     : {len(unique_sources)}")

print(f"📏 Avg Chars/Chunk  : {avg_chars:.1f}")
print(f"📖 Avg Words/Chunk  : {avg_words:.1f}")
print("=" * 70)

print(f"\n🚀 Starting embedding for {total_chunks} chunks...\n")


📊 DATASET SUMMARY
📄 Total Chunks      : 94
📝 Total Characters : 60,064
🔤 Total Words      : 6,965
📂 Source Files     : 1
📏 Avg Chars/Chunk  : 639.0
📖 Avg Words/Chunk  : 74.1

🚀 Starting embedding for 94 chunks...



In [6]:
# 1. Install Required Python Packages
!pip install -q pypdf langchain langchain-community python-dotenv pyyaml openai faiss-cpu


In [33]:
from openai import OpenAI
import os
import json
import time
import numpy as np
import faiss

# ============================================================
# Configuration
# ============================================================

MODEL = "text-embedding-3-small"
BATCH_SIZE = 32
MAX_RETRIES = 5
INITIAL_BACKOFF = 1.0


api_key = api_key

# Euron OpenAI-compatible endpoint
BASE_URL = "https://api.euron.one/api/v1/euri"

if not api_key:
    raise ValueError("API key is missing.")

client = OpenAI(
    api_key=api_key,
    base_url=BASE_URL,
)

# ============================================================
# Validate Chunks
# ============================================================

valid_chunks = []

for chunk in chunks_list:
    text = chunk.page_content.strip()

    if not text:
        continue

    # Skip unusually large chunks
    if len(text) > 30000:
        print(f"Skipping oversized chunk ({len(text)} chars)")
        continue

    valid_chunks.append(chunk)

print(f"\nEmbedding {len(valid_chunks)} chunks...\n")

# ============================================================
# Embedding Function
# ============================================================

def get_embeddings(texts):
    response = client.embeddings.create(
        model=MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]


def get_embeddings_with_retry(texts):
    backoff = INITIAL_BACKOFF

    for attempt in range(MAX_RETRIES):
        try:
            return get_embeddings(texts)

        except Exception as e:
            error = str(e)

            # Authentication errors should not be retried
            if "401" in error or "authentication" in error.lower():
                raise RuntimeError(f"Authentication failed:\n{error}")

            print(
                f"Retry {attempt + 1}/{MAX_RETRIES} "
                f"after error:\n{error}"
            )

            if attempt == MAX_RETRIES - 1:
                raise

            time.sleep(backoff)
            backoff *= 2


# ============================================================
# Generate Embeddings
# ============================================================

vector_database = []

total_batches = (len(valid_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

for batch_number, start in enumerate(
    range(0, len(valid_chunks), BATCH_SIZE), start=1
):

    batch = valid_chunks[start:start + BATCH_SIZE]
    texts = [chunk.page_content.strip() for chunk in batch]

    try:
        embeddings = get_embeddings_with_retry(texts)

        if len(embeddings) != len(batch):
            raise RuntimeError(
                f"Expected {len(batch)} embeddings, got {len(embeddings)}"
            )

        for index, (chunk, embedding) in enumerate(zip(batch, embeddings)):

            metadata = chunk.metadata or {}

            vector_database.append(
                {
                    "id": f"chunk_{start + index}",
                    "text": chunk.page_content.strip(),
                    "embedding": embedding,
                    "metadata": {
                        "source": metadata.get("source", "unknown"),
                        "page": metadata.get("page", 0),
                    },
                }
            )

        print(
            f"Batch {batch_number}/{total_batches} "
            f"completed "
            f"({len(vector_database)}/{len(valid_chunks)})"
        )

    except Exception as e:
        print(f"\nBatch {batch_number} failed:\n{e}\n")

# ============================================================
# Ensure Output Directory Exists
# ============================================================

os.makedirs(vector_out_dir, exist_ok=True)

# ============================================================
# Build FAISS Index
# ============================================================

if not vector_database:
    raise RuntimeError(
        "No embeddings were generated. Check your API key or API endpoint."
    )

print("\nBuilding FAISS index...")

embedding_matrix = np.array(
    [item["embedding"] for item in vector_database],
    dtype=np.float32,
)

# Normalize for cosine similarity
faiss.normalize_L2(embedding_matrix)

dimension = embedding_matrix.shape[1]

faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embedding_matrix)

faiss_path = os.path.join(vector_out_dir, "faiss_index.bin")

faiss.write_index(faiss_index, faiss_path)

print(f"FAISS index saved to:\n{faiss_path}")

# ============================================================
# Save JSON Database
# ============================================================

json_path = os.path.join(
    vector_out_dir,
    "vector_database.json",
)

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(vector_database, f, indent=2)

print(f"JSON database saved to:\n{json_path}")

# ============================================================
# Finished
# ============================================================

print("\n" + "=" * 60)
print(f"Successfully embedded {len(vector_database)} chunks.")
print("=" * 60)


Embedding 94 chunks...

Batch 1/3 completed (32/94)
Batch 2/3 completed (64/94)
Batch 3/3 completed (94/94)

Building FAISS index...
FAISS index saved to:
public/data/knowledge\faiss_index.bin
JSON database saved to:
public/data/knowledge\vector_database.json

Successfully embedded 94 chunks.
